# RBC Data

---

(HBN, NKI, PNC, BHRC, and CCNP)

### package imports and basic functions

---

In [1]:
import os
import gc
import sys
import glob
import shutil
import json
import random
import datetime
import importlib
import itertools
import numpy as np
from scipy import spatial
import scipy.sparse as sparse
import scipy.stats as stats
import pandas as pd
import nibabel as nib
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
from tqdm.auto import tqdm
from urllib.parse import urlparse
import requests
import zipfile
from pathlib import Path
import polars as pl
# import globus_sdk


In [2]:
%load_ext autoreload
%autoreload 2

# Path to add to src folder (to use local version of spectranorm)
sys.path.append(os.path.abspath("/mountpoint/code/projects/spectranorm/package/spectranorm/src/"))

from spectranorm import snm


In [3]:
class MyNumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        else:
            return super(MyEncoder, self).default(obj)


def ensure_dir(file_name):
    os.makedirs(os.path.dirname(file_name), exist_ok=True)
    return file_name


def list_dirs(path=os.getcwd()):
    files = glob.glob(os.path.join(path, '*'))
    files = [x for x in files if os.path.isdir(x)]
    return files


def file_exists(file_name, path_name=os.getcwd()):
    return os.path.isfile(os.path.join(path_name, file_name))


def write_json(json_obj, file_path):
    with open(file_path, 'w') as outfile:
        json.dump(json_obj, outfile, sort_keys=True, indent=4,
                  cls=MyNumpyEncoder)
    return json_obj


def load_json(file_path):
    with open(file_path, 'r') as infile:
        return json.load(infile)


def write_np(np_obj, file_path):
    with open(file_path, 'wb') as outfile:
        np.save(outfile, np_obj)


## Data access via DataLad

---


In [4]:
import datalad.api as dl


In [5]:
datasets = ["HBN", "NKI", "PNC", "BHRC", "CCNP"]


In [11]:
import os, sys
os.environ["PATH"] = "/mountpoint/code/projects/spectranorm/environment/spectranorm_env/bin:" + os.environ["PATH"]

!which git-annex


/mountpoint/code/projects/spectranorm/environment/spectranorm_env/bin/git-annex


In [13]:
# Let's first download demographies

# Clone relevant repositories via datalad
for dataset in datasets:
    repo_path = Path(f"/mountpoint/data/RBC/clones/{dataset}")
    dl.clone(
        source=f"https://github.com/ReproBrainChart/{dataset}_BIDS.git",
        path=repo_path,
    )

# Check here for more information:
# https://reprobrainchart.github.io/docs/get_data


[INFO] Attempting a clone into /mountpoint/data/RBC/clones/HBN 
[INFO] Attempting to clone from https://github.com/ReproBrainChart/HBN_BIDS.git to /mountpoint/data/RBC/clones/HBN 
[INFO] Start enumerating objects 
[INFO] Start counting objects 
[INFO] Start compressing objects 
[INFO] Start receiving objects 
[INFO] Start resolving deltas 
[INFO] Completed clone attempts for Dataset(/mountpoint/data/RBC/clones/HBN) 
[INFO] Remote origin not usable by git-annex; setting annex-ignore 
[INFO] https://github.com/ReproBrainChart/HBN_BIDS.git/config download failed: Not Found 


install(ok): /mountpoint/data/RBC/clones/HBN (dataset)


[INFO] Attempting a clone into /mountpoint/data/RBC/clones/NKI 
[INFO] Attempting to clone from https://github.com/ReproBrainChart/NKI_BIDS.git to /mountpoint/data/RBC/clones/NKI 
[INFO] Start enumerating objects 
[INFO] Start counting objects 
[INFO] Start compressing objects 
[INFO] Start receiving objects 
[INFO] Start resolving deltas 
[INFO] Completed clone attempts for Dataset(/mountpoint/data/RBC/clones/NKI) 
[INFO] Remote origin not usable by git-annex; setting annex-ignore 
[INFO] https://github.com/ReproBrainChart/NKI_BIDS.git/config download failed: Not Found 
[INFO] ssh: Could not resolve hostname sciget.pmacs.upenn.edu: Name or service not known 
[INFO] RIA store unavailable. -caused by- Failed to access ssh://sciget.pmacs.upenn.edu:/project/RBC/RIA/NKI/ria-layout-version -caused by- ConnectionOpenFailedError: 'ssh -fN -o ControlMaster=auto -o ControlPersist=15m -o ControlPath=/home/ubuntu/.cache/datalad/sockets/3eda6adc sciget.pmacs.upenn.edu' failed with exitcode 255 [Fa

install(ok): /mountpoint/data/RBC/clones/NKI (dataset)


[INFO] Attempting a clone into /mountpoint/data/RBC/clones/PNC 
[INFO] Attempting to clone from https://github.com/ReproBrainChart/PNC_BIDS.git to /mountpoint/data/RBC/clones/PNC 
[INFO] Start enumerating objects 
[INFO] Start counting objects 
[INFO] Start compressing objects 
[INFO] Start receiving objects 
[INFO] Start resolving deltas 
[INFO] Completed clone attempts for Dataset(/mountpoint/data/RBC/clones/PNC) 
[INFO] Remote origin not usable by git-annex; setting annex-ignore 
[INFO] https://github.com/ReproBrainChart/PNC_BIDS.git/config download failed: Not Found 


install(ok): /mountpoint/data/RBC/clones/PNC (dataset)


[INFO] Attempting a clone into /mountpoint/data/RBC/clones/BHRC 
[INFO] Attempting to clone from https://github.com/ReproBrainChart/BHRC_BIDS.git to /mountpoint/data/RBC/clones/BHRC 
[INFO] Start enumerating objects 
[INFO] Start counting objects 
[INFO] Start compressing objects 
[INFO] Start receiving objects 
[INFO] Start resolving deltas 
[INFO] Completed clone attempts for Dataset(/mountpoint/data/RBC/clones/BHRC) 
[INFO] Remote origin not usable by git-annex; setting annex-ignore 
[INFO] https://github.com/ReproBrainChart/BHRC_BIDS.git/config download failed: Not Found 
[INFO] ssh: Could not resolve hostname sciget.pmacs.upenn.edu: Name or service not known 
[INFO] RIA store unavailable. -caused by- Failed to access ssh://sciget.pmacs.upenn.edu:/project/RBC/RIA/BIDS/ria-layout-version -caused by- ConnectionOpenFailedError: 'ssh -fN -o ControlMaster=auto -o ControlPersist=15m -o ControlPath=/home/ubuntu/.cache/datalad/sockets/3eda6adc sciget.pmacs.upenn.edu' failed with exitcode 2

install(ok): /mountpoint/data/RBC/clones/BHRC (dataset)


[INFO] Attempting a clone into /mountpoint/data/RBC/clones/CCNP 
[INFO] Attempting to clone from https://github.com/ReproBrainChart/CCNP_BIDS.git to /mountpoint/data/RBC/clones/CCNP 
[INFO] Start enumerating objects 
[INFO] Start counting objects 
[INFO] Start compressing objects 
[INFO] Start receiving objects 
[INFO] Start resolving deltas 
[INFO] Completed clone attempts for Dataset(/mountpoint/data/RBC/clones/CCNP) 
[INFO] Remote origin not usable by git-annex; setting annex-ignore 
[INFO] https://github.com/ReproBrainChart/CCNP_BIDS.git/config download failed: Not Found 


install(ok): /mountpoint/data/RBC/clones/CCNP (dataset)


In [24]:
# Now load the participant.tsv file for every dataset

participant_dfs = {}
for dataset in datasets:
    participant_dfs[dataset] = pd.read_csv(
        f"/mountpoint/data/RBC/clones/{dataset}/study-{dataset}_desc-participants.tsv",
        sep="\t",
    )


In [ ]:
participant_dfs['HBN'].head()

In [ ]:
participant_dfs['NKI'].head()

In [ ]:
participant_dfs['PNC'].head()

In [ ]:
participant_dfs['BHRC'].head()

In [ ]:
participant_dfs['CCNP'].head()

In [20]:
# Let's clone the freesurfer repositories next:
for dataset in datasets:
    repo_path = Path(f"/mountpoint/data/RBC/clones/{dataset}_FreeSurfer")
    dl.clone(
        source=f"https://github.com/ReproBrainChart/{dataset}_FreeSurfer.git",
        path=repo_path,
    )

# Check here for more information:
# https://reprobrainchart.github.io/docs/get_data


[INFO] Attempting a clone into /mountpoint/data/RBC/clones/HBN_FreeSurfer 
[INFO] Attempting to clone from https://github.com/ReproBrainChart/HBN_FreeSurfer.git to /mountpoint/data/RBC/clones/HBN_FreeSurfer 
[INFO] Start enumerating objects 
[INFO] Start counting objects 
[INFO] Start compressing objects 
[INFO] Start receiving objects 
[INFO] Start resolving deltas 
[INFO] Completed clone attempts for Dataset(/mountpoint/data/RBC/clones/HBN_FreeSurfer) 
[INFO] Remote origin not usable by git-annex; setting annex-ignore 
[INFO] https://github.com/ReproBrainChart/HBN_FreeSurfer.git/config download failed: Not Found 
[INFO] RIA store unavailable. -caused by- file:///cbica/projects/RBC/freesurfer_stats/HBN/fs-tabulate/output_ria/ria-layout-version not found, self.ria_store_url: ria+file:///cbica/projects/RBC/freesurfer_stats/HBN/fs-tabulate/output_ria, self.store_base_pass: /cbica/projects/RBC/freesurfer_stats/HBN/fs-tabulate/output_ria, self.store_base_pass_push: None, path: <class 'path

install(ok): /mountpoint/data/RBC/clones/HBN_FreeSurfer (dataset)


[INFO] Attempting a clone into /mountpoint/data/RBC/clones/NKI_FreeSurfer 
[INFO] Attempting to clone from https://github.com/ReproBrainChart/NKI_FreeSurfer.git to /mountpoint/data/RBC/clones/NKI_FreeSurfer 
[INFO] Start enumerating objects 
[INFO] Start counting objects 
[INFO] Start compressing objects 
[INFO] Start receiving objects 
[INFO] Start resolving deltas 
[INFO] Completed clone attempts for Dataset(/mountpoint/data/RBC/clones/NKI_FreeSurfer) 
[INFO] Remote origin not usable by git-annex; setting annex-ignore 
[INFO] https://github.com/ReproBrainChart/NKI_FreeSurfer.git/config download failed: Not Found 
[INFO] RIA store unavailable. -caused by- file:///cbica/projects/RBC/freesurfer_stats/NKI/fs-tabulate/output_ria/ria-layout-version not found, self.ria_store_url: ria+file:///cbica/projects/RBC/freesurfer_stats/NKI/fs-tabulate/output_ria, self.store_base_pass: /cbica/projects/RBC/freesurfer_stats/NKI/fs-tabulate/output_ria, self.store_base_pass_push: None, path: <class 'path

install(ok): /mountpoint/data/RBC/clones/NKI_FreeSurfer (dataset)


[INFO] Attempting a clone into /mountpoint/data/RBC/clones/PNC_FreeSurfer 
[INFO] Attempting to clone from https://github.com/ReproBrainChart/PNC_FreeSurfer.git to /mountpoint/data/RBC/clones/PNC_FreeSurfer 
[INFO] Start enumerating objects 
[INFO] Start counting objects 
[INFO] Start compressing objects 
[INFO] Start receiving objects 
[INFO] Start resolving deltas 
[INFO] Completed clone attempts for Dataset(/mountpoint/data/RBC/clones/PNC_FreeSurfer) 
[INFO] Remote origin not usable by git-annex; setting annex-ignore 
[INFO] https://github.com/ReproBrainChart/PNC_FreeSurfer.git/config download failed: Not Found 
[INFO] RIA store unavailable. -caused by- file:///cbica/projects/RBC/freesurfer_stats/PNC/fs-tabulate/output_ria/ria-layout-version not found, self.ria_store_url: ria+file:///cbica/projects/RBC/freesurfer_stats/PNC/fs-tabulate/output_ria, self.store_base_pass: /cbica/projects/RBC/freesurfer_stats/PNC/fs-tabulate/output_ria, self.store_base_pass_push: None, path: <class 'path

install(ok): /mountpoint/data/RBC/clones/PNC_FreeSurfer (dataset)


[INFO] Attempting a clone into /mountpoint/data/RBC/clones/BHRC_FreeSurfer 
[INFO] Attempting to clone from https://github.com/ReproBrainChart/BHRC_FreeSurfer.git to /mountpoint/data/RBC/clones/BHRC_FreeSurfer 
[INFO] Start enumerating objects 
[INFO] Start counting objects 
[INFO] Start compressing objects 
[INFO] Start receiving objects 
[INFO] Start resolving deltas 
[INFO] Completed clone attempts for Dataset(/mountpoint/data/RBC/clones/BHRC_FreeSurfer) 
[INFO] Remote origin not usable by git-annex; setting annex-ignore 
[INFO] https://github.com/ReproBrainChart/BHRC_FreeSurfer.git/config download failed: Not Found 
[INFO] RIA store unavailable. -caused by- file:///cbica/projects/RBC/freesurfer_stats/BHRC/fs-tabulate/output_ria/ria-layout-version not found, self.ria_store_url: ria+file:///cbica/projects/RBC/freesurfer_stats/BHRC/fs-tabulate/output_ria, self.store_base_pass: /cbica/projects/RBC/freesurfer_stats/BHRC/fs-tabulate/output_ria, self.store_base_pass_push: None, path: <cla

install(ok): /mountpoint/data/RBC/clones/BHRC_FreeSurfer (dataset)


[INFO] Attempting a clone into /mountpoint/data/RBC/clones/CCNP_FreeSurfer 
[INFO] Attempting to clone from https://github.com/ReproBrainChart/CCNP_FreeSurfer.git to /mountpoint/data/RBC/clones/CCNP_FreeSurfer 
[INFO] Start enumerating objects 
[INFO] Start counting objects 
[INFO] Start compressing objects 
[INFO] Start receiving objects 
[INFO] Start resolving deltas 
[INFO] Completed clone attempts for Dataset(/mountpoint/data/RBC/clones/CCNP_FreeSurfer) 
[INFO] Remote origin not usable by git-annex; setting annex-ignore 
[INFO] https://github.com/ReproBrainChart/CCNP_FreeSurfer.git/config download failed: Not Found 
[INFO] RIA store unavailable. -caused by- file:///cbica/projects/RBC/freesurfer_stats/CCNP/fs-tabulate/output_ria/ria-layout-version not found, self.ria_store_url: ria+file:///cbica/projects/RBC/freesurfer_stats/CCNP/fs-tabulate/output_ria, self.store_base_pass: /cbica/projects/RBC/freesurfer_stats/CCNP/fs-tabulate/output_ria, self.store_base_pass_push: None, path: <cla

install(ok): /mountpoint/data/RBC/clones/CCNP_FreeSurfer (dataset)


In [25]:
# Now load the qc.tsv file for every dataset

qc_dfs = {}
for dataset in datasets:
    qc_dfs[dataset] = pd.read_csv(
        f"/mountpoint/data/RBC/clones/{dataset}_FreeSurfer/study-{dataset}_desc-T1_qc.tsv",
        sep="\t",
    )


In [ ]:
qc_dfs['HBN'].head()

In [ ]:
qc_dfs['NKI'].head()

In [ ]:
qc_dfs['PNC'].head()

In [ ]:
qc_dfs['BHRC'].head()

In [ ]:
qc_dfs['CCNP'].head()

## Testing datalad download for one subject:

---

In [ ]:
# Tring download for a single subject:
dataset = 'NKI'

# Exemplary subject ID
tmp_sub = participant_dfs[dataset]["participant_id"][0]
tmp_ses = participant_dfs[dataset]["session_id"][0]
print(tmp_sub, tmp_ses)


# Path to the cloned dataset
dataset_path = Path(f"/mountpoint/data/RBC/clones/{dataset}_FreeSurfer")

# Relative path within the dataset
file_to_get = dataset_path / f"freesurfer/sub-{tmp_sub}_ses-{tmp_ses}/sub-{tmp_sub}_ses-{tmp_ses}_freesurfer.tar.xz"
print(file_to_get)

# Fetch the content
res = dl.get(file_to_get, dataset=dataset_path)


In [ ]:
import tarfile
from pathlib import Path

tar_path = file_to_get

# Open the tar.xz file
with tarfile.open(tar_path, mode="r:xz") as tar:
    # List all files inside
    for member in tar.getmembers():
        print(member.name, member.size)  # name and size in bytes

In [ ]:
dl.drop(file_to_get, dataset=dataset_path)

## Extracting data: HBN

---

In [223]:
dataset = "HBN"

# Path to the cloned dataset
dataset_path = Path(f"/mountpoint/data/RBC/clones/{dataset}_FreeSurfer")

subjects = list(participant_dfs[dataset]["participant_id"])
len(subjects)


2611

In [ ]:
# ensuring there are no longitudinal data for HBN
participant_dfs[dataset]["participant_id"].value_counts()

In [99]:
import joblib
import tarfile, os, shutil
import numpy as np

items = [
    "lh.white", "rh.white",
    "lh.pial", "rh.pial",
    "lh.thickness", "rh.thickness",
    "lh.orig.nofix", "rh.orig.nofix",
    "lh.sphere.reg", "rh.sphere.reg",
]

def process_subject(idx, subject, dataset, dataset_path):
    sub_dir = f"{idx:02d}"[-2:]
    
    tar_path = dataset_path / f"freesurfer/sub-{subject}/sub-{subject}_freesurfer.tar.xz"
    freesurfer_directory = f"/mountpoint/data/RBC/snm_thickness/{dataset}/freesurfer/{subject}/"
    thickness_fslr_output = f"/mountpoint/data/normative/fs_LR_32k/{dataset}/{sub_dir}/{subject}.thickness.fslr.npy"

    if os.path.exists(thickness_fslr_output):
        return f"Skipping {subject}, already processed"

    if not tar_path.parent.exists():
        return f"Skipping {subject}, no TAR at {tar_path}"

    # Download
    dl.get(tar_path, dataset=dataset_path, result_renderer="disabled")

    # Extract needed files
    try:
        with tarfile.open(tar_path, mode="r:xz") as tar:
            for item in items:
                file_in_tar = f"sub-{subject}/surf/{item}"
                tar.extract(file_in_tar, path=freesurfer_directory)
    except tarfile.ReadError as e:
        return f"Skipping {subject}, corrupt TAR: {e}"

    # Drop the TAR
    dl.drop(tar_path, dataset=dataset_path, result_renderer="disabled")

    # Compute
    transformed_fslr_thickness = snm.utils.nitools.compute_fslr_thickness(
        os.path.join(freesurfer_directory, f"sub-{subject}")
    )
    np.save(
        ensure_dir(thickness_fslr_output),
        transformed_fslr_thickness.astype(np.float32)
    )

    # Cleanup
    shutil.rmtree(freesurfer_directory, ignore_errors=True)

    return f"Done {subject}"


In [ ]:
# Run in parallel
results = snm.utils.parallel.ParallelTqdm(n_jobs=32, total_tasks=len(subjects), desc=f"Computing fslr thickness for {dataset}",)(
    joblib.delayed(process_subject)(idx, subject, dataset, dataset_path)
    for idx, subject in enumerate(subjects)
)


In [55]:
# list of valid subjects
valid_subjects = {}

# empty dict to hold information
valid_subjects_dict = {}


In [ ]:
# list of valid subjects
valid_subjects[dataset] = [
    subject
    for idx, subject in enumerate(subjects)
    if Path(f"/mountpoint/data/normative/fs_LR_32k/{dataset}/{(idx%100):02d}/{subject}.thickness.fslr.npy").exists()
]

# empty dict to hold information
valid_subjects_dict[dataset] = {
    subject: {
        "unique_id": subject,
        "participant_id": subject,
        "session_id": "w1",
        "subject_index": idx,
    }
    for idx, subject in enumerate(subjects)
    if Path(f"/mountpoint/data/normative/fs_LR_32k/{dataset}/{(idx%100):02d}/{subject}.thickness.fslr.npy").exists()
}

len(valid_subjects_dict[dataset]), list(valid_subjects_dict[dataset].items())[:1]


In [ ]:
from contextlib import suppress

sex_encoder = {
    'Female': 'F',
    'Male': 'M'
}

# add information from participants.tsv
for _, row in tqdm(participant_dfs[dataset].iterrows()):
    key = row["participant_id"]
    if key in valid_subjects_dict[dataset]:
        if row['sex'] in sex_encoder:
            valid_subjects_dict[dataset][key]['age'] = row['age']
            valid_subjects_dict[dataset][key]['sex'] = sex_encoder[row['sex']]
            valid_subjects_dict[dataset][key]['site'] = row['study_site']
            valid_subjects_dict[dataset][key]['p-factor'] = row['p_factor_mcelroy_harmonized_all_samples']
        else:
            valid_subjects_dict[dataset].pop(key)

# add information from qc.tsv
for _, row in tqdm(qc_dfs[dataset].iterrows()):
    key = row["participant_id"]
    if key in valid_subjects_dict[dataset]:
        valid_subjects_dict[dataset][key]['qc_determination'] = row['qc_determination']
        valid_subjects_dict[dataset][key]['euler_no'] = row['euler']

# mean thickness and validity check
for key in tqdm(valid_subjects_dict[dataset]):
    idx = valid_subjects_dict[dataset][key]["subject_index"]
    subject = valid_subjects_dict[dataset][key]["unique_id"]
    valid_subjects_dict[dataset][key]["thickness"] = np.load(
        f"/mountpoint/data/normative/fs_LR_32k/{dataset}/{(idx%100):02d}/{subject}.thickness.fslr.npy",
    ).mean()
    valid_subjects_dict[dataset][key]["validity_check"] = (valid_subjects_dict[dataset][key]['qc_determination'] in ['Pass',])
    # valid_subjects_dict[dataset][key]["validity_check"] = (valid_subjects_dict[dataset][key]['qc_determination'] in ['Pass', 'Artifact'])

len(valid_subjects_dict[dataset]), list(valid_subjects_dict[dataset].items())[:1]


In [208]:
np.save(
    ensure_dir(f"/mountpoint/data/normative/datasets/{dataset}/subjects.npy"),
    np.array(valid_subjects[dataset])
)


In [209]:
import joblib

joblib.dump(valid_subjects_dict[dataset], ensure_dir(f"/mountpoint/data/normative/datasets/{dataset}/subjects.joblib"))


['/mountpoint/data/normative/datasets/HBN/subjects.joblib']

In [ ]:
import joblib

# Load the dictionary
valid_subjects_dict[dataset] = joblib.load(f"/mountpoint/data/normative/datasets/{dataset}/subjects.joblib")

len(valid_subjects_dict[dataset]), list(valid_subjects_dict[dataset].items())[:1]


In [ ]:
final_df = pd.DataFrame({
    'age': [valid_subjects_dict[dataset][key]["age"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'thickness': [valid_subjects_dict[dataset][key]["thickness"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'sex': [valid_subjects_dict[dataset][key]["sex"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'site': [valid_subjects_dict[dataset][key]["site"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'subject_ID': [valid_subjects_dict[dataset][key]["participant_id"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'euler_no': [valid_subjects_dict[dataset][key]["euler_no"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'subject_folder': [valid_subjects_dict[dataset][key]["unique_id"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'subject_index': [valid_subjects_dict[dataset][key]["subject_index"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
})
final_df['dataset'] = dataset
final_df.head(), final_df.shape


In [226]:
# randomly select only one timepoint per subject (cross-sectional sample)
final_df_subset = final_df.groupby("subject_ID", group_keys=False).sample(n=1, random_state=1234)

# # Only one site:
# final_df_subset.to_parquet(
#     ensure_dir(f'/mountpoint/data/normative/datasets/{dataset}/demography.parquet')
# )

# final_df_subset.shape

# Multiple sites:
# Keep only sites with at least 15 subjects
subjects_per_site = final_df_subset.groupby("site")["subject_ID"].nunique()
valid_sites = subjects_per_site[subjects_per_site >= 15].index

final_df_subset[final_df_subset["site"].isin(valid_sites)].to_parquet(
    ensure_dir(f'/mountpoint/data/normative/datasets/{dataset}/demography.parquet')
)

final_df_subset[final_df_subset["site"].isin(valid_sites)].shape


(1341, 9)

## Extracting data: NKI

---

In [222]:
dataset = "NKI"

# Path to the cloned dataset
dataset_path = Path(f"/mountpoint/data/RBC/clones/{dataset}_FreeSurfer")

subjects = list(participant_dfs[dataset]["participant_id"])

sessions = list(participant_dfs[dataset]["session_id"])

len(subjects)


2306

In [ ]:
# NKI has longitudinal follow ups
participant_dfs[dataset]["participant_id"].value_counts()

In [81]:
import joblib
import tarfile, os, shutil
import numpy as np

items = [
    "lh.white", "rh.white",
    "lh.pial", "rh.pial",
    "lh.thickness", "rh.thickness",
    "lh.orig.nofix", "rh.orig.nofix",
    "lh.sphere.reg", "rh.sphere.reg",
]

def process_subject_session(idx, subject, session, dataset, dataset_path):
    sub_dir = f"{idx:02d}"[-2:]
    
    tar_path = dataset_path / f"freesurfer/sub-{subject}_ses-{session}/sub-{subject}_ses-{session}_freesurfer.tar.xz"
    freesurfer_directory = f"/mountpoint/data/RBC/snm_thickness/{dataset}/freesurfer/sub-{subject}_ses-{session}/"
    thickness_fslr_output = f"/mountpoint/data/normative/fs_LR_32k/{dataset}/{sub_dir}/{subject}-{session}.thickness.fslr.npy"

    if os.path.exists(thickness_fslr_output):
        return f"Skipping {subject}-{session}, already processed"

    if not tar_path.parent.exists():
        return f"Skipping {subject}-{session}, no TAR at {tar_path}"

    # Download
    dl.get(tar_path, dataset=dataset_path, result_renderer="disabled")

    # Extract needed files
    try:
        with tarfile.open(tar_path, mode="r:xz") as tar:
            for item in items:
                file_in_tar = f"sub-{subject}_ses-{session}/surf/{item}"
                tar.extract(file_in_tar, path=freesurfer_directory)
    except tarfile.ReadError as e:
        return f"Skipping {subject}-{session}, corrupt TAR: {e}"

    # Drop the TAR
    dl.drop(tar_path, dataset=dataset_path, result_renderer="disabled")

    # Compute
    transformed_fslr_thickness = snm.utils.nitools.compute_fslr_thickness(
        os.path.join(freesurfer_directory, f"sub-{subject}_ses-{session}")
    )
    np.save(
        ensure_dir(thickness_fslr_output),
        transformed_fslr_thickness.astype(np.float32)
    )

    # Cleanup
    shutil.rmtree(freesurfer_directory, ignore_errors=True)

    return f"Done {subject}-{session}"


In [ ]:
# Run in parallel
results = snm.utils.parallel.ParallelTqdm(n_jobs=48, total_tasks=len(subjects), desc=f"Computing fslr thickness for {dataset}",)(
    joblib.delayed(process_subject_session)(idx, subject, sessions[idx], dataset, dataset_path)
    for idx, subject in enumerate(subjects)
)


In [ ]:
# list of valid subjects
valid_subjects[dataset] = [
    f"{subject}-{sessions[idx]}"
    for idx, subject in enumerate(subjects)
    if Path(f"/mountpoint/data/normative/fs_LR_32k/{dataset}/{(idx%100):02d}/{subject}-{sessions[idx]}.thickness.fslr.npy").exists()
]

# empty dict to hold information
valid_subjects_dict[dataset] = {
    f"{subject}-{sessions[idx]}": {
        "unique_id": f"{subject}-{sessions[idx]}",
        "participant_id": subject,
        "session_id": sessions[idx],
        "subject_index": idx,
    }
    for idx, subject in enumerate(subjects)
    if Path(f"/mountpoint/data/normative/fs_LR_32k/{dataset}/{(idx%100):02d}/{subject}-{sessions[idx]}.thickness.fslr.npy").exists()
}

len(valid_subjects_dict[dataset]), list(valid_subjects_dict[dataset].items())[:1]


In [ ]:
from contextlib import suppress

sex_encoder = {
    'Female': 'F',
    'Male': 'M'
}

# add information from participants.tsv
for _, row in tqdm(participant_dfs[dataset].iterrows()):
    key = f"{row['participant_id']}-{row['session_id']}"
    if key in valid_subjects_dict[dataset]:
        if row['sex'] in sex_encoder:
            valid_subjects_dict[dataset][key]['age'] = row['age']
            valid_subjects_dict[dataset][key]['sex'] = sex_encoder[row['sex']]
            valid_subjects_dict[dataset][key]['site'] = row['study_site']
            # valid_subjects_dict[dataset][key]['p-factor'] = row['p_factor_mcelroy_harmonized_all_samples']
        else:
            valid_subjects_dict[dataset].pop(key)

# add information from qc.tsv
for _, row in tqdm(qc_dfs[dataset].iterrows()):
    key = f"{row['participant_id']}-{row['session_id']}"
    if key in valid_subjects_dict[dataset]:
        valid_subjects_dict[dataset][key]['qc_determination'] = row['qc_determination']
        valid_subjects_dict[dataset][key]['euler_no'] = row['euler']

# mean thickness and validity check
for key in tqdm(valid_subjects_dict[dataset]):
    idx = valid_subjects_dict[dataset][key]["subject_index"]
    subject = valid_subjects_dict[dataset][key]["unique_id"]
    valid_subjects_dict[dataset][key]["thickness"] = np.load(
        f"/mountpoint/data/normative/fs_LR_32k/{dataset}/{(idx%100):02d}/{subject}.thickness.fslr.npy",
    ).mean()
    valid_subjects_dict[dataset][key]["validity_check"] = (valid_subjects_dict[dataset][key]['qc_determination'] in ['Pass', 'Artifact'])

len(valid_subjects_dict[dataset]), list(valid_subjects_dict[dataset].items())[:1]


In [153]:
np.save(
    ensure_dir(f"/mountpoint/data/normative/datasets/{dataset}/subjects.npy"),
    np.array(valid_subjects[dataset])
)


In [154]:
import joblib

joblib.dump(valid_subjects_dict[dataset], ensure_dir(f"/mountpoint/data/normative/datasets/{dataset}/subjects.joblib"))


['/mountpoint/data/normative/datasets/NKI/subjects.joblib']

In [ ]:
import joblib

# Load the dictionary
valid_subjects_dict[dataset] = joblib.load(f"/mountpoint/data/normative/datasets/{dataset}/subjects.joblib")

len(valid_subjects_dict[dataset]), list(valid_subjects_dict[dataset].items())[:1]


In [ ]:
final_df = pd.DataFrame({
    'age': [valid_subjects_dict[dataset][key]["age"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'thickness': [valid_subjects_dict[dataset][key]["thickness"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'sex': [valid_subjects_dict[dataset][key]["sex"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    # 'site': [valid_subjects_dict[dataset][key]["site"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'subject_ID': [valid_subjects_dict[dataset][key]["participant_id"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'euler_no': [valid_subjects_dict[dataset][key]["euler_no"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'subject_folder': [valid_subjects_dict[dataset][key]["unique_id"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'subject_index': [valid_subjects_dict[dataset][key]["subject_index"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
})
final_df['dataset'] = dataset
final_df.head(), final_df.shape


In [157]:
# randomly select only one timepoint per subject (cross-sectional sample)
final_df_subset = final_df.groupby("subject_ID", group_keys=False).sample(n=1, random_state=1234)

# Only one site:
final_df_subset.to_parquet(
    ensure_dir(f'/mountpoint/data/normative/datasets/{dataset}/demography.parquet')
)

final_df_subset.shape

# # Multiple sites:
# # Keep only sites with at least 15 subjects
# subjects_per_site = final_df_subset.groupby("site")["subject_ID"].nunique()
# valid_sites = subjects_per_site[subjects_per_site >= 15].index

# final_df_subset[final_df_subset["site"].isin(valid_sites)].to_parquet(
#     ensure_dir(f'/mountpoint/data/normative/datasets/{dataset}/demography.parquet')
# )

# final_df_subset[final_df_subset["site"].isin(valid_sites)].shape


(1303, 8)

## Extracting data: PNC

---

In [220]:
dataset = "PNC"

# Path to the cloned dataset
dataset_path = Path(f"/mountpoint/data/RBC/clones/{dataset}_FreeSurfer")

subjects = list(participant_dfs[dataset]["participant_id"])

sessions = list(participant_dfs[dataset]["session_id"])

len(subjects)


1601

In [ ]:
# ensuring there are no longitudinal data for PNC
participant_dfs[dataset]["participant_id"].value_counts()

In [ ]:
# Run in parallel
results = snm.utils.parallel.ParallelTqdm(n_jobs=56, total_tasks=len(subjects), desc=f"Computing fslr thickness for {dataset}",)(
    joblib.delayed(process_subject)(idx, subject, dataset, dataset_path)
    for idx, subject in enumerate(subjects)
)


In [ ]:
# list of valid subjects
valid_subjects[dataset] = [
    subject
    for idx, subject in enumerate(subjects)
    if Path(f"/mountpoint/data/normative/fs_LR_32k/{dataset}/{(idx%100):02d}/{subject}.thickness.fslr.npy").exists()
]

# empty dict to hold information
valid_subjects_dict[dataset] = {
    subject: {
        "unique_id": subject,
        "participant_id": subject,
        "session_id": "w1",
        "subject_index": idx,
    }
    for idx, subject in enumerate(subjects)
    if Path(f"/mountpoint/data/normative/fs_LR_32k/{dataset}/{(idx%100):02d}/{subject}.thickness.fslr.npy").exists()
}

len(valid_subjects_dict[dataset]), list(valid_subjects_dict[dataset].items())[:1]


In [ ]:
from contextlib import suppress

sex_encoder = {
    'Female': 'F',
    'Male': 'M'
}

# add information from participants.tsv
for _, row in tqdm(participant_dfs[dataset].iterrows()):
    key = row["participant_id"]
    if key in valid_subjects_dict[dataset]:
        if row['sex'] in sex_encoder:
            valid_subjects_dict[dataset][key]['age'] = row['age']
            valid_subjects_dict[dataset][key]['sex'] = sex_encoder[row['sex']]
            valid_subjects_dict[dataset][key]['site'] = row['study_site']
            valid_subjects_dict[dataset][key]['p-factor'] = row['p_factor_mcelroy_harmonized_all_samples']
        else:
            valid_subjects_dict[dataset].pop(key)

# add information from qc.tsv
for _, row in tqdm(qc_dfs[dataset].iterrows()):
    key = row["participant_id"]
    if key in valid_subjects_dict[dataset]:
        valid_subjects_dict[dataset][key]['qc_determination'] = row['qc_determination']
        valid_subjects_dict[dataset][key]['euler_no'] = row['euler']

# mean thickness and validity check
for key in tqdm(valid_subjects_dict[dataset]):
    idx = valid_subjects_dict[dataset][key]["subject_index"]
    subject = valid_subjects_dict[dataset][key]["unique_id"]
    valid_subjects_dict[dataset][key]["thickness"] = np.load(
        f"/mountpoint/data/normative/fs_LR_32k/{dataset}/{(idx%100):02d}/{subject}.thickness.fslr.npy",
    ).mean()
    valid_subjects_dict[dataset][key]["validity_check"] = (valid_subjects_dict[dataset][key]['qc_determination'] in ['Pass', 'Artifact'])

len(valid_subjects_dict[dataset]), list(valid_subjects_dict[dataset].items())[:1]


In [180]:
np.save(
    ensure_dir(f"/mountpoint/data/normative/datasets/{dataset}/subjects.npy"),
    np.array(valid_subjects[dataset])
)


In [181]:
import joblib

joblib.dump(valid_subjects_dict[dataset], ensure_dir(f"/mountpoint/data/normative/datasets/{dataset}/subjects.joblib"))


['/mountpoint/data/normative/datasets/PNC/subjects.joblib']

In [ ]:
import joblib

# Load the dictionary
valid_subjects_dict[dataset] = joblib.load(f"/mountpoint/data/normative/datasets/{dataset}/subjects.joblib")

len(valid_subjects_dict[dataset]), list(valid_subjects_dict[dataset].items())[:1]


In [ ]:
final_df = pd.DataFrame({
    'age': [valid_subjects_dict[dataset][key]["age"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'thickness': [valid_subjects_dict[dataset][key]["thickness"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'sex': [valid_subjects_dict[dataset][key]["sex"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    # 'site': [valid_subjects_dict[dataset][key]["site"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'subject_ID': [valid_subjects_dict[dataset][key]["participant_id"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'euler_no': [valid_subjects_dict[dataset][key]["euler_no"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'subject_folder': [valid_subjects_dict[dataset][key]["unique_id"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'subject_index': [valid_subjects_dict[dataset][key]["subject_index"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
})
final_df['dataset'] = dataset
final_df.head(), final_df.shape


In [184]:
# randomly select only one timepoint per subject (cross-sectional sample)
final_df_subset = final_df.groupby("subject_ID", group_keys=False).sample(n=1, random_state=1234)

# Only one site:
final_df_subset.to_parquet(
    ensure_dir(f'/mountpoint/data/normative/datasets/{dataset}/demography.parquet')
)

final_df_subset.shape

# # Multiple sites:
# # Keep only sites with at least 15 subjects
# subjects_per_site = final_df_subset.groupby("site")["subject_ID"].nunique()
# valid_sites = subjects_per_site[subjects_per_site >= 15].index

# final_df_subset[final_df_subset["site"].isin(valid_sites)].to_parquet(
#     ensure_dir(f'/mountpoint/data/normative/datasets/{dataset}/demography.parquet')
# )

# final_df_subset[final_df_subset["site"].isin(valid_sites)].shape


(1584, 8)

## Extracting data: BHRC

---

In [215]:
dataset = "BHRC"

# Path to the cloned dataset
dataset_path = Path(f"/mountpoint/data/RBC/clones/{dataset}_FreeSurfer")

subjects = list(participant_dfs[dataset]["participant_id"])

sessions = list(participant_dfs[dataset]["session_id"])

len(subjects)


907

In [ ]:
# BHRC has longitudinal follow ups
participant_dfs[dataset]["participant_id"].value_counts()

In [117]:
import joblib
import tarfile, os, shutil
import numpy as np

items = [
    "lh.white", "rh.white",
    "lh.pial", "rh.pial",
    "lh.thickness", "rh.thickness",
    "lh.orig.nofix", "rh.orig.nofix",
    "lh.sphere.reg", "rh.sphere.reg",
]

def process_subject_session(idx, subject, session, dataset, dataset_path):
    sub_dir = f"{idx:02d}"[-2:]
    
    tar_path = dataset_path / f"freesurfer/sub-{subject}_ses-{session}/sub-{subject}_ses-{session}_freesurfer.tar.xz"
    freesurfer_directory = f"/mountpoint/data/RBC/snm_thickness/{dataset}/freesurfer/sub-{subject}_ses-{session}/"
    thickness_fslr_output = f"/mountpoint/data/normative/fs_LR_32k/{dataset}/{sub_dir}/{subject}-{session}.thickness.fslr.npy"

    if os.path.exists(thickness_fslr_output):
        return f"Skipping {subject}-{session}, already processed"

    if not tar_path.parent.exists():
        return f"Skipping {subject}-{session}, no TAR at {tar_path}"

    # Download
    dl.get(tar_path, dataset=dataset_path, result_renderer="disabled")

    # Extract needed files
    try:
        with tarfile.open(tar_path, mode="r:xz") as tar:
            # Get all members (file paths inside the tar)
            members = tar.getnames()
            # Grab the top-level directory (first component of the first member)
            topdir = members[0].split("/")[0]
            for item in items:
                # file_in_tar = f"sub-{subject}_ses-{session}/surf/{item}"
                file_in_tar = f"{topdir}/surf/{item}"
                tar.extract(file_in_tar, path=freesurfer_directory)
    except tarfile.ReadError as e:
        return f"Skipping {subject}-{session}, corrupt TAR: {e}"

    # Drop the TAR
    dl.drop(tar_path, dataset=dataset_path, result_renderer="disabled")

    # Compute
    transformed_fslr_thickness = snm.utils.nitools.compute_fslr_thickness(
        # os.path.join(freesurfer_directory, f"sub-{subject}_ses-{session}")
        os.path.join(freesurfer_directory, topdir)
    )
    np.save(
        ensure_dir(thickness_fslr_output),
        transformed_fslr_thickness.astype(np.float32)
    )

    # Cleanup
    shutil.rmtree(freesurfer_directory, ignore_errors=True)

    return f"Done {subject}-{session}"


In [ ]:
# Run in parallel
results = snm.utils.parallel.ParallelTqdm(n_jobs=48, total_tasks=len(subjects), desc=f"Computing fslr thickness for {dataset}",)(
    joblib.delayed(process_subject_session)(idx, subject, sessions[idx], dataset, dataset_path)
    for idx, subject in enumerate(subjects)
)


In [ ]:
# list of valid subjects
valid_subjects[dataset] = [
    f"{subject}-{sessions[idx]}"
    for idx, subject in enumerate(subjects)
    if Path(f"/mountpoint/data/normative/fs_LR_32k/{dataset}/{(idx%100):02d}/{subject}-{sessions[idx]}.thickness.fslr.npy").exists()
]

# empty dict to hold information
valid_subjects_dict[dataset] = {
    f"{subject}-{sessions[idx]}": {
        "unique_id": f"{subject}-{sessions[idx]}",
        "participant_id": subject,
        "session_id": sessions[idx],
        "subject_index": idx,
    }
    for idx, subject in enumerate(subjects)
    if Path(f"/mountpoint/data/normative/fs_LR_32k/{dataset}/{(idx%100):02d}/{subject}-{sessions[idx]}.thickness.fslr.npy").exists()
}

len(valid_subjects_dict[dataset]), list(valid_subjects_dict[dataset].items())[:1]


In [ ]:
from contextlib import suppress

sex_encoder = {
    'Female': 'F',
    'Male': 'M'
}

# add information from participants.tsv
for _, row in tqdm(participant_dfs[dataset].iterrows()):
    key = f"{row['participant_id']}-{row['session_id']}"
    if key in valid_subjects_dict[dataset]:
        if row['sex'] in sex_encoder:
            valid_subjects_dict[dataset][key]['age'] = row['age']
            valid_subjects_dict[dataset][key]['sex'] = sex_encoder[row['sex']]
            valid_subjects_dict[dataset][key]['site'] = row['study_site']
            valid_subjects_dict[dataset][key]['p-factor'] = row['p_factor_mcelroy_harmonized_all_samples']
        else:
            valid_subjects_dict[dataset].pop(key)

# add information from qc.tsv
for _, row in tqdm(qc_dfs[dataset].iterrows()):
    key = f"{row['participant_id']}-{row['session_id']}"
    if key in valid_subjects_dict[dataset]:
        valid_subjects_dict[dataset][key]['qc_determination'] = row['qc_determination']
        valid_subjects_dict[dataset][key]['euler_no'] = row['euler']

# mean thickness and validity check
for key in tqdm(valid_subjects_dict[dataset]):
    idx = valid_subjects_dict[dataset][key]["subject_index"]
    subject = valid_subjects_dict[dataset][key]["unique_id"]
    valid_subjects_dict[dataset][key]["thickness"] = np.load(
        f"/mountpoint/data/normative/fs_LR_32k/{dataset}/{(idx%100):02d}/{subject}.thickness.fslr.npy",
    ).mean()
    valid_subjects_dict[dataset][key]["validity_check"] = (valid_subjects_dict[dataset][key]['qc_determination'] in ['Pass', 'Artifact'])

len(valid_subjects_dict[dataset]), list(valid_subjects_dict[dataset].items())[:1]


In [190]:
np.save(
    ensure_dir(f"/mountpoint/data/normative/datasets/{dataset}/subjects.npy"),
    np.array(valid_subjects[dataset])
)


In [191]:
import joblib

joblib.dump(valid_subjects_dict[dataset], ensure_dir(f"/mountpoint/data/normative/datasets/{dataset}/subjects.joblib"))


['/mountpoint/data/normative/datasets/BHRC/subjects.joblib']

In [ ]:
import joblib

# Load the dictionary
valid_subjects_dict[dataset] = joblib.load(f"/mountpoint/data/normative/datasets/{dataset}/subjects.joblib")

len(valid_subjects_dict[dataset]), list(valid_subjects_dict[dataset].items())[:1]


In [ ]:
final_df = pd.DataFrame({
    'age': [valid_subjects_dict[dataset][key]["age"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'thickness': [valid_subjects_dict[dataset][key]["thickness"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'sex': [valid_subjects_dict[dataset][key]["sex"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'site': [valid_subjects_dict[dataset][key]["site"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'subject_ID': [valid_subjects_dict[dataset][key]["participant_id"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'euler_no': [valid_subjects_dict[dataset][key]["euler_no"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'subject_folder': [valid_subjects_dict[dataset][key]["unique_id"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'subject_index': [valid_subjects_dict[dataset][key]["subject_index"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
})
final_df['dataset'] = dataset
final_df.head(), final_df.shape


In [218]:
final_df['site'].value_counts()

site
BHRC-2    561
BHRC-1    261
Name: count, dtype: int64

In [219]:
# randomly select only one timepoint per subject (cross-sectional sample)
final_df_subset = final_df.groupby("subject_ID", group_keys=False).sample(n=1, random_state=1234)

# # Only one site:
# final_df_subset.to_parquet(
#     ensure_dir(f'/mountpoint/data/normative/datasets/{dataset}/demography.parquet')
# )

# final_df_subset.shape

# Multiple sites:
# Keep only sites with at least 15 subjects
subjects_per_site = final_df_subset.groupby("site")["subject_ID"].nunique()
valid_sites = subjects_per_site[subjects_per_site >= 15].index

final_df_subset[final_df_subset["site"].isin(valid_sites)].to_parquet(
    ensure_dir(f'/mountpoint/data/normative/datasets/{dataset}/demography.parquet')
)

final_df_subset[final_df_subset["site"].isin(valid_sites)].shape


(580, 9)

## Extracting data: CCNP

---

In [213]:
dataset = "CCNP"

# Path to the cloned dataset
dataset_path = Path(f"/mountpoint/data/RBC/clones/{dataset}_FreeSurfer")

subjects = list(participant_dfs[dataset]["participant_id"])

sessions = list(participant_dfs[dataset]["session_id"])

len(subjects)


195

In [ ]:
# ensuring there are no longitudinal data for CCNP
participant_dfs[dataset]["participant_id"].value_counts()

In [131]:
# Run in parallel
results = snm.utils.parallel.ParallelTqdm(n_jobs=56, total_tasks=len(subjects), desc=f"Computing fslr thickness for {dataset}",)(
    joblib.delayed(process_subject)(idx, subject, dataset, dataset_path)
    for idx, subject in enumerate(subjects)
)


Computing fslr thickness for CCNP:   0%|          | 0/195 [00:00<?, ?tasks/s]

In [ ]:
# list of valid subjects
valid_subjects[dataset] = [
    subject
    for idx, subject in enumerate(subjects)
    if Path(f"/mountpoint/data/normative/fs_LR_32k/{dataset}/{(idx%100):02d}/{subject}.thickness.fslr.npy").exists()
]

# empty dict to hold information
valid_subjects_dict[dataset] = {
    subject: {
        "unique_id": subject,
        "participant_id": subject,
        "session_id": "w1",
        "subject_index": idx,
    }
    for idx, subject in enumerate(subjects)
    if Path(f"/mountpoint/data/normative/fs_LR_32k/{dataset}/{(idx%100):02d}/{subject}.thickness.fslr.npy").exists()
}

len(valid_subjects_dict[dataset]), list(valid_subjects_dict[dataset].items())[:1]


In [ ]:
from contextlib import suppress

sex_encoder = {
    'Female': 'F',
    'Male': 'M'
}

# add information from participants.tsv
for _, row in tqdm(participant_dfs[dataset].iterrows()):
    key = row["participant_id"]
    if key in valid_subjects_dict[dataset]:
        if row['sex'] in sex_encoder:
            valid_subjects_dict[dataset][key]['age'] = row['age']
            valid_subjects_dict[dataset][key]['sex'] = sex_encoder[row['sex']]
            valid_subjects_dict[dataset][key]['site'] = row['study_site']
            valid_subjects_dict[dataset][key]['p-factor'] = row['p_factor_mcelroy_harmonized_all_samples']
        else:
            valid_subjects_dict[dataset].pop(key)

# add information from qc.tsv
for _, row in tqdm(qc_dfs[dataset].iterrows()):
    key = row["participant_id"]
    if key in valid_subjects_dict[dataset]:
        valid_subjects_dict[dataset][key]['qc_determination'] = row['qc_determination']
        valid_subjects_dict[dataset][key]['euler_no'] = row['euler']

# mean thickness and validity check
for key in tqdm(valid_subjects_dict[dataset]):
    idx = valid_subjects_dict[dataset][key]["subject_index"]
    subject = valid_subjects_dict[dataset][key]["unique_id"]
    valid_subjects_dict[dataset][key]["thickness"] = np.load(
        f"/mountpoint/data/normative/fs_LR_32k/{dataset}/{(idx%100):02d}/{subject}.thickness.fslr.npy",
    ).mean()
    valid_subjects_dict[dataset][key]["validity_check"] = (valid_subjects_dict[dataset][key]['qc_determination'] in ['Pass', 'Artifact'])

len(valid_subjects_dict[dataset]), list(valid_subjects_dict[dataset].items())[:1]


In [199]:
np.save(
    ensure_dir(f"/mountpoint/data/normative/datasets/{dataset}/subjects.npy"),
    np.array(valid_subjects[dataset])
)


In [200]:
import joblib

joblib.dump(valid_subjects_dict[dataset], ensure_dir(f"/mountpoint/data/normative/datasets/{dataset}/subjects.joblib"))


['/mountpoint/data/normative/datasets/CCNP/subjects.joblib']

In [ ]:
import joblib

# Load the dictionary
valid_subjects_dict[dataset] = joblib.load(f"/mountpoint/data/normative/datasets/{dataset}/subjects.joblib")

len(valid_subjects_dict[dataset]), list(valid_subjects_dict[dataset].items())[:1]


In [ ]:
final_df = pd.DataFrame({
    'age': [valid_subjects_dict[dataset][key]["age"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'thickness': [valid_subjects_dict[dataset][key]["thickness"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'sex': [valid_subjects_dict[dataset][key]["sex"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    # 'site': [valid_subjects_dict[dataset][key]["site"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'subject_ID': [valid_subjects_dict[dataset][key]["participant_id"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'euler_no': [valid_subjects_dict[dataset][key]["euler_no"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'subject_folder': [valid_subjects_dict[dataset][key]["unique_id"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
    'subject_index': [valid_subjects_dict[dataset][key]["subject_index"] for key in valid_subjects_dict[dataset] if valid_subjects_dict[dataset][key]["validity_check"]],
})
final_df['dataset'] = dataset
final_df.head(), final_df.shape


In [203]:
# randomly select only one timepoint per subject (cross-sectional sample)
final_df_subset = final_df.groupby("subject_ID", group_keys=False).sample(n=1, random_state=1234)

# Only one site:
final_df_subset.to_parquet(
    ensure_dir(f'/mountpoint/data/normative/datasets/{dataset}/demography.parquet')
)

final_df_subset.shape

# # Multiple sites:
# # Keep only sites with at least 15 subjects
# subjects_per_site = final_df_subset.groupby("site")["subject_ID"].nunique()
# valid_sites = subjects_per_site[subjects_per_site >= 15].index

# final_df_subset[final_df_subset["site"].isin(valid_sites)].to_parquet(
#     ensure_dir(f'/mountpoint/data/normative/datasets/{dataset}/demography.parquet')
# )

# final_df_subset[final_df_subset["site"].isin(valid_sites)].shape


(194, 8)

## Scratch

---

In [204]:
participant_dfs.keys()

dict_keys(['HBN', 'NKI', 'PNC', 'BHRC', 'CCNP'])